# Placebo-Window Falsification Audit for Founder Exit\n\nThis notebook demonstrates `eval.py`, the **Placebo-Window Falsification and Robustness Audit** for a founder-exit authority-diffusion / OSS-survival experiment.\n\nGiven a per-repository event table (founder pre-departure diffusion scores + 18-month survival labels), `eval.py` runs four independent checks:\n\n1. **`placebo_test`** — compares the true pre-departure window's survival-rate lift against a *placebo* (randomly-relocated) window's lift, across multiple random seeds, to falsify the claim that the true window specifically matters.\n2. **`stratified_robustness`** — reruns the effect separately per language / popularity bucket and checks for heterogeneity (Simpson's-paradox-style ecosystem dominance).\n3. **`pipeline_validity`** — sanity-checks the reimplementation against Avelino et al.'s published aggregate statistics (TFDD rate, TF=1 share, unconditioned survival rate) with Wilson confidence intervals.\n4. **`calibration`** — bootstraps a predicted-probability calibration curve, Brier score, per-coefficient CIs, and AUC for the survival logistic regression.\n\n**Data note:** at the time this evaluation artifact was finalized, the upstream experiment had not yet produced its `method_out.json`, so the real `eval_out.json` reports every check as `UNAVAILABLE` (a documented pipeline gap, not a negative result). To demonstrate the evaluation logic itself, this notebook runs the *exact same, unmodified* `eval.py` functions against a small **synthetic** dataset (`mini_demo_data.json`) built to match the upstream schema `eval.py` expects — so every check below actually executes and returns `COMPUTED`.

In [ ]:
import subprocess, sys\ndef _pip(*a): subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *a])\n\n# loguru is NOT pre-installed on Colab -- always install\n_pip('loguru==0.7.3')\n\n# numpy, pandas, scipy -- pre-installed on Colab; install locally only, at Colab's exact versions\nif 'google.colab' not in sys.modules:\n    _pip('numpy==2.0.2', 'pandas==2.2.2', 'scipy==1.16.3', 'matplotlib==3.10.0')

In [ ]:
# Original imports from eval.py, plus matplotlib for the results visualization at the end.\nfrom __future__ import annotations\n\nimport json\nfrom typing import Any\n\nimport numpy as np\nimport pandas as pd\nfrom loguru import logger\n\nimport matplotlib.pyplot as plt\n\nlogger.remove()\nlogger.add(sys.stdout, level=\"INFO\", format=\"{time:HH:mm:ss}|{level:<7}|{message}\")

## Load the demo data\n\n`mini_demo_data.json` is a small synthetic dataset shaped like the upstream `method_out.json` (an `exp_gen_sol_out`-style `datasets/examples` payload with `metadata_`-prefixed fields) plus a `method_summary` block, exactly what `eval.py`'s `events_to_dataframe()` and `run_pipeline_validity()` expect. We try the GitHub-hosted copy first, then fall back to the local file (works both on Colab and locally).

In [ ]:
GITHUB_DATA_URL = \"https://raw.githubusercontent.com/ai-inventor-papers/ai-invention-24ffbe-pre-departure-bus-factor-diffusion/main/round-1/evaluation-1/demo/mini_demo_data.json\"\nimport os\n\ndef load_data():\n    try:\n        import urllib.request\n        with urllib.request.urlopen(GITHUB_DATA_URL) as response:\n            return json.loads(response.read().decode())\n    except Exception:\n        pass\n    if os.path.exists(\"mini_demo_data.json\"):\n        with open(\"mini_demo_data.json\") as f:\n            return json.load(f)\n    raise FileNotFoundError(\"Could not load mini_demo_data.json\")

In [ ]:
data = load_data()\nprint(data[\"note\"])\nmethod_out = data[\"method_out\"]\nmethod_summary = data[\"method_summary\"]\nprint(f\"\\nLoaded {len(method_out['datasets'][0]['examples'])} synthetic per-project event records\")

## Config\n\nAll tunable parameters from the original `eval.py`, gathered here. Original values (`RNG_SEEDS = [1234, 5678, 9012]`, `N_BOOT = 2000`, `N_BOOT_CALIB = 1000`) are cheap enough on this tiny 60-row demo dataset to run as-is within the runtime budget, so no scaling-down was needed.

In [ ]:
RNG_SEEDS = [1234, 5678, 9012]  # >=3 seeds for placebo seed-sensitivity, recorded for reproducibility\nN_BOOT = 2000  # bootstrap resamples for CIs (reduced from 5000 to stay within CPU budget on 4 cores)\nN_BOOT_CALIB = 1000  # calibration bootstrap, per plan (>=1000)\n\nAVELINO_TFDD_RATE = 315 / 1932  # ~0.163\nAVELINO_TF1_SHARE = 0.66\nAVELINO_TFDD_SURVIVAL = 128 / 315  # ~0.406

## Parse the per-project event table\n\n`events_to_dataframe()` (unchanged from `eval.py`) converts the `method_out.json`-style `datasets/examples` payload into a flat `DataFrame`, stripping the `metadata_` prefix, deriving `survived` from the `output` field, and renaming a couple of columns to what the rest of the script expects.

In [ ]:
def _events_from_exp_gen_sol_out(method_out: dict[str, Any]) -> pd.DataFrame | None:\n    \"\"\"Extract the per-event table from the actual upstream schema: an\n    exp_gen_sol_out-style {\"datasets\": [{\"examples\": [...]}]} payload where each\n    example carries `metadata_*`-prefixed fields plus an `output` string of\n    \"survived\" / \"did_not_survive\" (the label is NOT a metadata_ field).\"\"\"\n    datasets = method_out.get(\"datasets\")\n    if not isinstance(datasets, list) or not datasets:\n        return None\n    examples = datasets[0].get(\"examples\")\n    if not isinstance(examples, list) or not examples:\n        return None\n    rows = []\n    for ex in examples:\n        if not isinstance(ex, dict) or \"metadata_repo\" not in ex:\n            continue  # skip diagnostic placeholder rows (e.g. \"no_events\")\n        row = {k[len(\"metadata_\"):]: v for k, v in ex.items() if k.startswith(\"metadata_\")}\n        row[\"survived\"] = 1 if ex.get(\"output\") == \"survived\" else 0\n        rows.append(row)\n    if not rows:\n        return None\n    df = pd.DataFrame(rows)\n\n    # Normalize to the column names the rest of this evaluation expects.\n    rename_map = {\n        \"founder_share_pre_departure\": \"founder_share\",\n        \"n_diffused_owners_pre_departure\": \"n_diffused_owners\",\n    }\n    df = df.rename(columns={k: v for k, v in rename_map.items() if k in df.columns})\n    if \"censored\" in df.columns:\n        df = df[~df[\"censored\"].astype(bool)].copy()\n    if \"stars\" in df.columns:\n        df[\"log_stars\"] = np.log1p(pd.to_numeric(df[\"stars\"], errors=\"coerce\"))\n    if \"forks\" in df.columns:\n        df[\"log_forks\"] = np.log1p(pd.to_numeric(df[\"forks\"], errors=\"coerce\"))\n    if \"devs_at_tfdd\" in df.columns and \"n_contributors\" not in df.columns:\n        df[\"n_contributors\"] = df[\"devs_at_tfdd\"]\n    if \"stars\" in df.columns and \"popularity_bucket\" not in df.columns:\n        try:\n            df[\"popularity_bucket\"] = pd.qcut(\n                pd.to_numeric(df[\"stars\"], errors=\"coerce\"), q=3, labels=[\"low\", \"mid\", \"high\"], duplicates=\"drop\"\n            ).astype(str)\n        except ValueError:\n            pass  # too few distinct star values to form 3 buckets; stratification falls back to language only\n    return df\n\n\ndef events_to_dataframe(method_out: dict[str, Any] | None) -> pd.DataFrame | None:\n    \"\"\"Extract the per-event record table from method_out.json, tolerant of the\n    exact upstream schema variant it was written in.\"\"\"\n    if method_out is None:\n        return None\n    # Preferred / actual upstream shape: exp_gen_sol_out-style datasets/examples.\n    df = _events_from_exp_gen_sol_out(method_out)\n    if df is not None:\n        return df\n    # Fallback: a flat list of event dicts under one of these keys (in case a\n    # different experiment run wrote a simpler shape).\n    candidates = [\"per_event_records\", \"events\", \"tfdd_events\", \"records\", \"founder_tfdd_events\"]\n    for key in candidates:\n        if key in method_out and isinstance(method_out[key], list) and len(method_out[key]) > 0:\n            return pd.DataFrame(method_out[key])\n    return None\n\n\ndf = events_to_dataframe(method_out)\nlogger.info(f\"Loaded {len(df)} per-project event records from upstream experiment output\")\ndf.head()

## Statistical helpers\n\nGeneric helpers used by all four checks: Wilson score intervals, a generic bootstrap CI routine, Benjamini-Hochberg FDR correction, Cohen's h, Brier score, a rank-based AUC, and a from-scratch IRLS logistic regression (no external stats dependency beyond numpy/scipy).

In [ ]:
def wilson_ci(successes: int, n: int, z: float = 1.96) -> tuple[float, float, float]:\n    \"\"\"Wilson score interval for a binomial proportion. Returns (point, lo, hi).\"\"\"\n    if n == 0:\n        return (float(\"nan\"), float(\"nan\"), float(\"nan\"))\n    p = successes / n\n    denom = 1 + z**2 / n\n    center = (p + z**2 / (2 * n)) / denom\n    half = (z * np.sqrt(p * (1 - p) / n + z**2 / (4 * n**2))) / denom\n    return (p, max(0.0, center - half), min(1.0, center + half))\n\n\ndef bootstrap_ci(values: np.ndarray, stat_fn, n_boot: int, seed: int) -> tuple[float, float, float]:\n    \"\"\"Generic bootstrap: returns (point estimate, 2.5%, 97.5%) for stat_fn(values).\"\"\"\n    rng = np.random.default_rng(seed)\n    if len(values) == 0:\n        return (float(\"nan\"), float(\"nan\"), float(\"nan\"))\n    point = stat_fn(values)\n    n = len(values)\n    boots = np.empty(n_boot)\n    for b in range(n_boot):\n        idx = rng.integers(0, n, size=n)\n        boots[b] = stat_fn(values[idx])\n    lo, hi = np.percentile(boots, [2.5, 97.5])\n    return (float(point), float(lo), float(hi))\n\n\ndef benjamini_hochberg(pvals: dict[str, float], alpha: float = 0.05) -> dict[str, float]:\n    \"\"\"Return BH-adjusted p-values keyed identically to the input dict.\"\"\"\n    items = sorted(pvals.items(), key=lambda kv: kv[1])\n    m = len(items)\n    adjusted = {}\n    prev = 1.0\n    for rank, (k, p) in enumerate(reversed(items), start=1):\n        i = m - rank + 1\n        val = min(prev, p * m / i)\n        prev = val\n        adjusted[k] = val\n    return adjusted\n\n\ndef cohens_h(p1: float, p2: float) -> float:\n    \"\"\"Cohen's h effect size for the difference between two proportions.\"\"\"\n    return 2 * np.arcsin(np.sqrt(p1)) - 2 * np.arcsin(np.sqrt(p2))\n\n\ndef brier_score(y_true: np.ndarray, y_prob: np.ndarray) -> float:\n    return float(np.mean((y_prob - y_true) ** 2))\n\n\ndef auc_score(y_true: np.ndarray, y_prob: np.ndarray) -> float:\n    \"\"\"Mann-Whitney U based AUC, no sklearn dependency needed for ties handling.\"\"\"\n    pos = y_prob[y_true == 1]\n    neg = y_prob[y_true == 0]\n    if len(pos) == 0 or len(neg) == 0:\n        return float(\"nan\")\n    # Rank-based AUC (handles ties by average rank)\n    all_scores = np.concatenate([pos, neg])\n    order = np.argsort(all_scores)\n    ranks = np.empty_like(order, dtype=float)\n    ranks[order] = np.arange(1, len(all_scores) + 1)\n    # average tie ranks\n    _, inv, counts = np.unique(all_scores, return_inverse=True, return_counts=True)\n    sum_ranks_per_val = np.zeros(len(counts))\n    np.add.at(sum_ranks_per_val, inv, ranks)\n    avg_rank_per_val = sum_ranks_per_val / counts\n    ranks = avg_rank_per_val[inv]\n    rank_pos_sum = ranks[: len(pos)].sum()\n    n1, n0 = len(pos), len(neg)\n    u = rank_pos_sum - n1 * (n1 + 1) / 2\n    return float(u / (n1 * n0))\n\n\ndef logistic_regression_irls(X: np.ndarray, y: np.ndarray, max_iter: int = 100, tol: float = 1e-8):\n    \"\"\"Minimal IRLS logistic regression (no external dep beyond numpy).\n\n    X must already include an intercept column. Returns (coefs, cov_matrix) or\n    (None, None) on failure (e.g. singular Hessian / quasi-separation).\n    \"\"\"\n    n, p = X.shape\n    beta = np.zeros(p)\n    for _ in range(max_iter):\n        eta = X @ beta\n        eta = np.clip(eta, -30, 30)\n        mu = 1 / (1 + np.exp(-eta))\n        w = mu * (1 - mu)\n        w = np.clip(w, 1e-8, None)\n        z = eta + (y - mu) / w\n        WX = X * w[:, None]\n        try:\n            hessian = X.T @ WX\n            beta_new = np.linalg.solve(hessian, X.T @ (w * z))\n        except np.linalg.LinAlgError:\n            return None, None\n        if np.max(np.abs(beta_new - beta)) < tol:\n            beta = beta_new\n            break\n        beta = beta_new\n    eta = np.clip(X @ beta, -30, 30)\n    mu = 1 / (1 + np.exp(-eta))\n    w = np.clip(mu * (1 - mu), 1e-8, None)\n    try:\n        cov = np.linalg.inv(X.T @ (X * w[:, None]))\n    except np.linalg.LinAlgError:\n        cov = np.full((p, p), np.nan)\n    return beta, cov\n\n\ndef wald_pvalues(beta: np.ndarray, cov: np.ndarray) -> np.ndarray:\n    from scipy import stats\n\n    se = np.sqrt(np.clip(np.diag(cov), 0, None))\n    with np.errstate(divide=\"ignore\", invalid=\"ignore\"):\n        z = beta / se\n    return 2 * (1 - stats.norm.cdf(np.abs(z)))

## Check 1: Placebo-window falsification\n\nCompares the **true** pre-departure window's survival-rate lift (survival rate of high-diffusion projects minus low-diffusion projects, split at the median `founder_share`) against the **placebo** window's lift, across multiple seeds. If the true lift sits clearly above the placebo distribution, that's evidence the true window specifically carries signal (not just any random pre-TFDD window).

In [ ]:
def run_placebo_falsification(df: pd.DataFrame, gaps: list[str]) -> dict[str, Any]:\n    \"\"\"Reconstruct the placebo/shuffle test comparing true vs random-window effects.\n\n    Requires: per-project founder_share / n_diffused_owners for the TRUE window,\n    a survival label, and EITHER (a) precomputed placebo_founder_share /\n    placebo_n_diffused_owners from the upstream experiment (Stage 7 of its\n    pseudocode), or (b) a full per-window time series to draw placebo windows\n    from ourselves. If neither is present we cannot fabricate a window series\n    (explicitly disallowed by the artifact plan) and report UNAVAILABLE.\n    \"\"\"\n    result: dict[str, Any] = {\"status\": \"UNAVAILABLE\", \"seeds\": RNG_SEEDS}\n\n    required_true = {\"founder_share\", \"n_diffused_owners\", \"survived\"}\n    if df is None or not required_true.issubset(df.columns):\n        gaps.append(\n            \"placebo_test: upstream event table missing one of \"\n            f\"{sorted(required_true)}; cannot run true-window statistics at all.\"\n        )\n        return result\n\n    has_placebo_precomputed = {\"placebo_founder_share\", \"placebo_n_diffused_owners\"}.issubset(df.columns)\n    has_window_series = \"pre_tfdd_window_series\" in df.columns or \"window_series\" in df.columns\n\n    if not has_placebo_precomputed and not has_window_series:\n        gaps.append(\n            \"placebo_test: neither precomputed placebo_founder_share/\"\n            \"placebo_n_diffused_owners columns nor a per-project pre-TFDD window \"\n            \"time series were present in the upstream event table. Per the artifact \"\n            \"plan's explicit fallback instruction, a placebo window series was NOT \"\n            \"fabricated. Falsification check (success_criteria #3) is UNAVAILABLE \"\n            \"this run; only Steps 4-6 (stratification / pipeline-validity / \"\n            \"calibration) could execute on whatever fields ARE present.\"\n        )\n        return result\n\n    df = df.dropna(subset=[\"founder_share\", \"n_diffused_owners\", \"survived\"]).copy()\n    df[\"survived\"] = df[\"survived\"].astype(int)\n\n    def group_lift(sub: pd.DataFrame, share_col: str) -> float:\n        \"\"\"Survival-rate lift: high-diffusion (low founder share) minus low-diffusion group.\"\"\"\n        lo = sub[sub[share_col] < sub[share_col].median()]\n        hi = sub[sub[share_col] >= sub[share_col].median()]\n        if len(lo) == 0 or len(hi) == 0:\n            return float(\"nan\")\n        return float(lo[\"survived\"].mean() - hi[\"survived\"].mean())\n\n    true_lift_point, true_lift_lo, true_lift_hi = bootstrap_ci(\n        df.index.values, lambda idx: group_lift(df.loc[idx], \"founder_share\"), N_BOOT, seed=RNG_SEEDS[0]\n    )\n\n    seed_results = []\n    if has_placebo_precomputed:\n        placebo_df = df.dropna(subset=[\"placebo_founder_share\", \"placebo_n_diffused_owners\"])\n        for seed in RNG_SEEDS:\n            # single precomputed draw: reuse it under each seed label for seed-sensitivity\n            # reporting, since the upstream only stored one placebo draw per project.\n            lift_p, lo_p, hi_p = bootstrap_ci(\n                placebo_df.index.values,\n                lambda idx: group_lift(placebo_df.loc[idx], \"placebo_founder_share\"),\n                N_BOOT,\n                seed=seed,\n            )\n            seed_results.append({\"seed\": seed, \"placebo_lift\": lift_p, \"ci_lo\": lo_p, \"ci_hi\": hi_p})\n        gaps.append(\n            \"placebo_test: upstream provided only ONE precomputed placebo draw per \"\n            \"project (not a full window series), so seed-sensitivity here reflects \"\n            \"bootstrap resampling variance under different seeds applied to the SAME \"\n            \"draw, not independent re-draws of the placebo window itself. This is a \"\n            \"weaker seed-sensitivity check than the artifact plan specifies.\"\n        )\n    else:\n        # has_window_series\n        series_col = \"pre_tfdd_window_series\" if \"pre_tfdd_window_series\" in df.columns else \"window_series\"\n        for seed in RNG_SEEDS:\n            rng = np.random.default_rng(seed)\n            placebo_rows = []\n            for _, row in df.iterrows():\n                windows = row[series_col]\n                if not isinstance(windows, list) or len(windows) == 0:\n                    continue\n                choice = windows[int(rng.integers(0, len(windows)))]\n                placebo_rows.append({\n                    \"placebo_founder_share\": choice.get(\"founder_share\"),\n                    \"placebo_n_diffused_owners\": choice.get(\"n_diffused_owners\"),\n                    \"survived\": row[\"survived\"],\n                })\n            pdf = pd.DataFrame(placebo_rows).dropna()\n            if len(pdf) == 0:\n                continue\n            lift_p, lo_p, hi_p = bootstrap_ci(\n                pdf.index.values,\n                lambda idx: group_lift(pdf.loc[idx], \"placebo_founder_share\"),\n                N_BOOT,\n                seed=seed,\n            )\n            seed_results.append({\"seed\": seed, \"placebo_lift\": lift_p, \"ci_lo\": lo_p, \"ci_hi\": hi_p})\n\n    if not seed_results:\n        gaps.append(\"placebo_test: placebo data present but produced 0 usable rows after cleaning.\")\n        return result\n\n    placebo_lifts = np.array([s[\"placebo_lift\"] for s in seed_results if not np.isnan(s[\"placebo_lift\"])])\n    if len(placebo_lifts) == 0:\n        gaps.append(\"placebo_test: all placebo lift estimates were NaN.\")\n        return result\n\n    diff = true_lift_point - float(np.mean(placebo_lifts))\n    # Permutation-style test: is the true effect outside the empirical placebo distribution?\n    ci_excludes_zero = not (true_lift_lo <= 0 <= true_lift_hi) and (true_lift_lo > np.max(placebo_lifts))\n    ci_overlap = not (true_lift_hi < np.min(placebo_lifts) or true_lift_lo > np.max(placebo_lifts))\n\n    if ci_excludes_zero and true_lift_point > np.max(placebo_lifts):\n        verdict = \"PASS\"\n    elif true_lift_point > float(np.mean(placebo_lifts)) and ci_overlap:\n        verdict = \"WEAK\"\n    else:\n        verdict = \"FAIL\"\n\n    result = {\n        \"status\": \"COMPUTED\",\n        \"n_projects\": int(len(df)),\n        \"true_window_survival_lift\": {\"point\": true_lift_point, \"ci95\": [true_lift_lo, true_lift_hi]},\n        \"placebo_survival_lift_by_seed\": seed_results,\n        \"placebo_lift_mean_across_seeds\": float(np.mean(placebo_lifts)),\n        \"true_minus_placebo_diff\": diff,\n        \"ci_overlap\": bool(ci_overlap),\n        \"verdict\": verdict,\n        \"seeds\": RNG_SEEDS,\n    }\n    return result\n\n\ngaps: list[str] = []\nplacebo_result = run_placebo_falsification(df, gaps)\nprint(json.dumps(placebo_result, indent=2, default=str))

## Check 2: Stratified robustness\n\nReruns the same survival-rate effect separately per `language` and `popularity_bucket`, flags underpowered strata (n<10), and computes a simple heterogeneity check (range of stratum effects vs the pooled CI width) to detect Simpson's-paradox-style ecosystem dominance.

In [ ]:
def run_stratified_robustness(df: pd.DataFrame, gaps: list[str]) -> dict[str, Any]:\n    required = {\"founder_share\", \"survived\"}\n    if df is None or not required.issubset(df.columns):\n        gaps.append(\"stratified_robustness: missing founder_share/survived columns; UNAVAILABLE.\")\n        return {\"status\": \"UNAVAILABLE\"}\n\n    df = df.dropna(subset=[\"founder_share\", \"survived\"]).copy()\n    df[\"survived\"] = df[\"survived\"].astype(int)\n\n    strata_cols = [c for c in [\"language\", \"popularity_bucket\", \"star_bucket\"] if c in df.columns]\n    if not strata_cols:\n        gaps.append(\n            \"stratified_robustness: no language/popularity_bucket columns found in the \"\n            \"upstream event table; cannot stratify. Reporting pooled effect only.\"\n        )\n        strata_cols = []\n\n    def effect(sub: pd.DataFrame) -> float:\n        lo = sub[sub[\"founder_share\"] < sub[\"founder_share\"].median()]\n        hi = sub[sub[\"founder_share\"] >= sub[\"founder_share\"].median()]\n        if len(lo) == 0 or len(hi) == 0:\n            return float(\"nan\")\n        return float(lo[\"survived\"].mean() - hi[\"survived\"].mean())\n\n    pooled_point, pooled_lo, pooled_hi = bootstrap_ci(\n        df.index.values, lambda idx: effect(df.loc[idx]), N_BOOT, seed=RNG_SEEDS[0]\n    )\n\n    strata_results = []\n    MIN_N = 10\n    for col in strata_cols:\n        for level, sub in df.groupby(col):\n            underpowered = len(sub) < MIN_N\n            if len(sub) < 4:\n                strata_results.append({\n                    \"stratum_col\": col, \"level\": str(level), \"n\": int(len(sub)),\n                    \"underpowered\": True, \"effect\": None, \"ci95\": None,\n                    \"note\": \"n<4, too small even to bootstrap\",\n                })\n                continue\n            pt, lo_, hi_ = bootstrap_ci(sub.index.values, lambda idx: effect(sub.loc[idx]), N_BOOT, seed=RNG_SEEDS[0])\n            strata_results.append({\n                \"stratum_col\": col, \"level\": str(level), \"n\": int(len(sub)),\n                \"underpowered\": bool(underpowered), \"effect\": pt, \"ci95\": [lo_, hi_],\n            })\n\n    # Heterogeneity: range of stratum effects vs pooled CI width, and simple Cochran's Q\n    valid_effects = [s[\"effect\"] for s in strata_results if s[\"effect\"] is not None and not np.isnan(s[\"effect\"])]\n    heterogeneity = {}\n    if len(valid_effects) >= 2:\n        eff_range = float(max(valid_effects) - min(valid_effects))\n        pooled_ci_width = float(pooled_hi - pooled_lo)\n        heterogeneity = {\n            \"effect_range_across_strata\": eff_range,\n            \"pooled_ci_width\": pooled_ci_width,\n            \"range_exceeds_pooled_ci\": bool(eff_range > pooled_ci_width),\n            \"n_strata_compared\": len(valid_effects),\n        }\n    else:\n        heterogeneity = {\"note\": \"fewer than 2 valid strata effects; heterogeneity check UNAVAILABLE\"}\n\n    return {\n        \"status\": \"COMPUTED\",\n        \"pooled_effect\": {\"point\": pooled_point, \"ci95\": [pooled_lo, pooled_hi]},\n        \"strata\": strata_results,\n        \"min_n_threshold\": MIN_N,\n        \"heterogeneity_check\": heterogeneity,\n    }\n\n\nstrat_result = run_stratified_robustness(df, gaps)\nprint(json.dumps(strat_result, indent=2, default=str))

## Check 3: Pipeline-validity sanity check vs Avelino et al.\n\nSanity-checks the reimplementation against Avelino et al.'s published aggregate statistics (TFDD rate ~16%, TF=1 share 66%, unconditioned TFDD survival 41%) using Wilson 95% CIs and a PASS/CONCERN flag inside a 1.5x relative-distance band. `method_summary` here comes from the synthetic `mini_demo_data.json`'s `method_summary` block (standing in for the upstream `results/method_summary.json`).

In [ ]:
def run_pipeline_validity(\n    method_out: dict[str, Any] | None, df: pd.DataFrame | None, gaps: list[str],\n    method_summary: dict[str, Any] | None = None,\n) -> dict[str, Any]:\n    checks: dict[str, Any] = {}\n    summary = method_summary or {}\n\n    def flag(name: str, point: float, lo: float, hi: float, reference: float) -> dict[str, Any]:\n        rel_dist = abs(point - reference) / reference if reference else float(\"inf\")\n        ci_contains = lo <= reference <= hi\n        passed = ci_contains or rel_dist <= 1.5\n        return {\n            \"point_estimate\": point, \"ci95\": [lo, hi], \"avelino_reference\": reference,\n            \"relative_distance\": rel_dist, \"flag\": \"PASS\" if passed else \"CONCERN\",\n        }\n\n    # (a) fraction of projects with >=1 TFDD. The upstream pipeline only ever records\n    # founder-only (strict, TF=1) and TF<=2 (relaxed) TFDD events -- it never counts\n    # TFDDs of any TF-set size, so \"n_repos_with_tfdd\" in Avelino et al.'s exact sense\n    # does not exist upstream. We use the RELAXED (TF<=2) count over n_repos_processed\n    # as the closest available proxy (an underestimate of the true any-TF-size rate,\n    # since TF=3+ TFDDs are invisible to this pipeline by construction) and label it\n    # explicitly as a proxy rather than a like-for-like reproduction.\n    n_processed = summary.get(\"n_repos_processed\")\n    n_relaxed = summary.get(\"n_founder_tfdd_events_relaxed\")\n    n_strict = summary.get(\"n_founder_tfdd_events_strict\")\n    if n_processed and n_relaxed is not None:\n        p, lo, hi = wilson_ci(int(n_relaxed), int(n_processed))\n        checks[\"tfdd_rate\"] = flag(\"tfdd_rate\", p, lo, hi, AVELINO_TFDD_RATE)\n        checks[\"tfdd_rate\"][\"proxy_caveat\"] = (\n            \"Upstream tracks only TF<=2 TFDDs (relaxed definition), not TFDDs of any \"\n            \"TF-set size as in Avelino et al.; this is a lower-bound proxy for the \"\n            \"true any-size TFDD rate, so a below-reference point estimate is expected \"\n            \"even with a correct implementation.\"\n        )\n    else:\n        gaps.append(\n            \"pipeline_validity/tfdd_rate: results/method_summary.json missing \"\n            \"n_repos_processed and/or n_founder_tfdd_events_relaxed; UNAVAILABLE.\"\n        )\n        checks[\"tfdd_rate\"] = {\"status\": \"UNAVAILABLE\"}\n\n    # (b) fraction of TFDDs at TF=1 (founder-only): proxy as strict / relaxed, i.e.\n    # among TF<=2 TFDDs, what share are exactly TF=1. This is NOT Avelino et al.'s\n    # exact \"share of ALL TFDDs (any TF size) that occur at TF=1\" -- their denominator\n    # includes TF=2,3,4... events this pipeline never detects -- so we report it as an\n    # informative but non-equivalent proxy rather than silently treating it as the\n    # same statistic.\n    if n_strict is not None and n_relaxed:\n        p, lo, hi = wilson_ci(int(n_strict), int(n_relaxed))\n        checks[\"tf1_share\"] = flag(\"tf1_share\", p, lo, hi, AVELINO_TF1_SHARE)\n        checks[\"tf1_share\"][\"proxy_caveat\"] = (\n            \"Computed as strict(TF=1) / relaxed(TF<=2), NOT strict / all-TFDDs-of-\"\n            \"any-size as in Avelino et al. -- the pipeline's own pseudocode only ever \"\n            \"detects founder-only or TF<=2 events, so the true denominator (TFDDs \"\n            \"with a larger initial TF-set) is structurally unmeasured by this \"\n            \"experiment. Treat this as directional evidence only, not a strict \"\n            \"replication of the 66% figure.\"\n        )\n    else:\n        gaps.append(\n            \"pipeline_validity/tf1_share: results/method_summary.json missing \"\n            \"n_founder_tfdd_events_strict and/or n_founder_tfdd_events_relaxed; \"\n            \"UNAVAILABLE. Note even with these fields present, this pipeline \"\n            \"structurally cannot reproduce Avelino et al.'s exact tf1_share \"\n            \"definition -- see the proxy_caveat this check would otherwise attach.\"\n        )\n        checks[\"tf1_share\"] = {\"status\": \"UNAVAILABLE\"}\n\n    # (c) unconditioned survival rate among founder-only (strict) TFDD events --\n    # this one IS directly comparable to Avelino et al.'s 41%, since both are\n    # \"P(survive 18mo | TFDD occurred)\" on an uncensored sample.\n    strict_surv = summary.get(\"strict_unconditioned_survival\") or {}\n    if strict_surv.get(\"survival_rate\") is not None and strict_surv.get(\"n_uncensored\"):\n        p = float(strict_surv[\"survival_rate\"])\n        n = int(strict_surv[\"n_uncensored\"])\n        k = round(p * n)\n        _, lo, hi = wilson_ci(k, n)\n        checks[\"unconditioned_survival_rate\"] = flag(\"unconditioned_survival_rate\", p, lo, hi, AVELINO_TFDD_SURVIVAL)\n    elif df is not None and \"survived\" in df.columns and len(df) > 0:\n        sub = df.dropna(subset=[\"survived\"])\n        n = int(len(sub))\n        if n > 0:\n            k = int(sub[\"survived\"].astype(int).sum())\n            p, lo, hi = wilson_ci(k, n)\n            checks[\"unconditioned_survival_rate\"] = flag(\n                \"unconditioned_survival_rate\", p, lo, hi, AVELINO_TFDD_SURVIVAL\n            )\n        else:\n            checks[\"unconditioned_survival_rate\"] = {\"status\": \"UNAVAILABLE\"}\n    else:\n        gaps.append(\n            \"pipeline_validity/unconditioned_survival_rate: no per-event survival \"\n            \"labels (from method_out.json) or precomputed strict_unconditioned_survival \"\n            \"(from method_summary.json) found; UNAVAILABLE.\"\n        )\n        checks[\"unconditioned_survival_rate\"] = {\"status\": \"UNAVAILABLE\"}\n\n    checks[\"caveat\"] = (\n        \"This evaluation's corpus is a founder-only, stratified-sampled subset \"\n        \"(6 languages x 3 popularity strata, target ~40/language per the experiment \"\n        \"plan) rather than Avelino et al.'s full top-500-per-language corpus (n=1932), \"\n        \"so some divergence from their published aggregates is EXPECTED and does not \"\n        \"by itself indicate a reimplementation bug; only a large divergence outside \"\n        \"the 1.5x relative-distance band is flagged CONCERN.\"\n    )\n    return checks\n\n\nvalidity_result = run_pipeline_validity(method_out, df, gaps, method_summary=method_summary)\nprint(json.dumps(validity_result, indent=2, default=str))

## Check 4: Regression calibration\n\nFits the from-scratch IRLS logistic regression of `survived` on the available predictors (`founder_share`, `n_diffused_owners`, `log_stars`, `log_forks`, `n_contributors`), then bootstraps a predicted-probability-decile calibration curve, Brier score, per-coefficient 95% CIs, and AUC.

In [ ]:
def run_calibration(df: pd.DataFrame, gaps: list[str]) -> dict[str, Any]:\n    predictor_cols = [c for c in [\"founder_share\", \"n_diffused_owners\", \"log_stars\", \"log_forks\", \"n_contributors\"] if df is not None and c in df.columns]\n    if df is None or \"survived\" not in df.columns or len(predictor_cols) == 0:\n        gaps.append(\n            \"calibration: missing survived label or all candidate predictor columns \"\n            \"(founder_share/n_diffused_owners/log_stars/log_forks/n_contributors); \"\n            \"UNAVAILABLE.\"\n        )\n        return {\"status\": \"UNAVAILABLE\"}\n\n    sub = df.dropna(subset=predictor_cols + [\"survived\"]).copy()\n    if len(sub) < 15:\n        gaps.append(\n            f\"calibration: only {len(sub)} complete rows available (need >=15 for a \"\n            \"stable logistic fit + bootstrap); UNAVAILABLE.\"\n        )\n        return {\"status\": \"UNAVAILABLE\", \"n_available\": int(len(sub))}\n\n    y = sub[\"survived\"].astype(int).to_numpy()\n    Xraw = sub[predictor_cols].to_numpy(dtype=float)\n    Xstd = (Xraw - Xraw.mean(axis=0)) / (Xraw.std(axis=0) + 1e-9)\n    X = np.column_stack([np.ones(len(sub)), Xstd])\n\n    beta, cov = logistic_regression_irls(X, y)\n    if beta is None:\n        gaps.append(\"calibration: logistic regression failed to converge (singular Hessian, likely quasi-separation).\")\n        return {\"status\": \"FAILED_TO_CONVERGE\", \"n_available\": int(len(sub))}\n\n    pvals = wald_pvalues(beta, cov)\n    coef_names = [\"intercept\"] + predictor_cols\n    pval_dict = {name: float(p) for name, p in zip(coef_names, pvals)}\n    bh = benjamini_hochberg({k: v for k, v in pval_dict.items() if k != \"intercept\"})\n\n    eta = np.clip(X @ beta, -30, 30)\n    p_hat = 1 / (1 + np.exp(-eta))\n\n    brier = brier_score(y, p_hat)\n    auc_pt, auc_lo, auc_hi = bootstrap_ci(\n        np.arange(len(y)), lambda idx: auc_score(y[idx], p_hat[idx]), N_BOOT_CALIB, seed=RNG_SEEDS[0]\n    )\n\n    # Bootstrap coefficient CIs\n    rng = np.random.default_rng(RNG_SEEDS[0])\n    n = len(y)\n    boot_coefs = []\n    for _ in range(N_BOOT_CALIB):\n        idx = rng.integers(0, n, size=n)\n        b, _ = logistic_regression_irls(X[idx], y[idx])\n        if b is not None:\n            boot_coefs.append(b)\n    coef_ci = {}\n    if boot_coefs:\n        boot_arr = np.array(boot_coefs)\n        for i, name in enumerate(coef_names):\n            lo_c, hi_c = np.percentile(boot_arr[:, i], [2.5, 97.5])\n            coef_ci[name] = {\"point\": float(beta[i]), \"ci95\": [float(lo_c), float(hi_c)], \"wald_p\": pval_dict[name]}\n    else:\n        gaps.append(\"calibration: all bootstrap resamples failed to converge; coefficient CIs UNAVAILABLE.\")\n\n    # Calibration curve: predicted-probability deciles vs observed survival rate\n    deciles = pd.qcut(p_hat, q=min(10, len(np.unique(p_hat))), duplicates=\"drop\")\n    calib_df = pd.DataFrame({\"decile\": deciles, \"p_hat\": p_hat, \"y\": y})\n    calib_curve = (\n        calib_df.groupby(\"decile\", observed=True)\n        .agg(mean_predicted=(\"p_hat\", \"mean\"), observed_rate=(\"y\", \"mean\"), n=(\"y\", \"size\"))\n        .reset_index(drop=True)\n        .to_dict(orient=\"records\")\n    )\n\n    return {\n        \"status\": \"COMPUTED\",\n        \"n\": int(len(sub)),\n        \"predictor_cols\": predictor_cols,\n        \"coefficients\": coef_ci,\n        \"bh_adjusted_pvalues\": bh,\n        \"brier_score\": brier,\n        \"auc\": {\"point\": auc_pt, \"ci95\": [auc_lo, auc_hi]},\n        \"calibration_curve_deciles\": calib_curve,\n        \"n_bootstrap\": N_BOOT_CALIB,\n    }\n\n\ncalib_result = run_calibration(df, gaps)\nprint(json.dumps(calib_result, indent=2, default=str))

## Results summary\n\nOverall verdict is taken straight from the placebo test (falsification is the headline `success_criteria #3` check); a readable table of the four checks' key numbers, plus two plots: true-vs-placebo survival lift by seed, and the bootstrapped calibration curve.

In [ ]:
overall_verdict = placebo_result[\"verdict\"] if placebo_result.get(\"status\") == \"COMPUTED\" else \"UNDETERMINED_PIPELINE_GAP\"\nprint(f\"Overall verdict: {overall_verdict}\\n\")\n\nsummary_rows = [\n    (\"placebo_test\", placebo_result.get(\"status\"), placebo_result.get(\"verdict\", \"-\")),\n    (\"stratified_robustness\", strat_result.get(\"status\"), f\"{len(strat_result.get('strata', []))} strata\" if strat_result.get(\"status\") == \"COMPUTED\" else \"-\"),\n    (\"pipeline_validity.tfdd_rate\", validity_result.get(\"tfdd_rate\", {}).get(\"flag\", validity_result.get(\"tfdd_rate\", {}).get(\"status\")), round(validity_result.get(\"tfdd_rate\", {}).get(\"point_estimate\", float(\"nan\")), 3)),\n    (\"pipeline_validity.tf1_share\", validity_result.get(\"tf1_share\", {}).get(\"flag\", validity_result.get(\"tf1_share\", {}).get(\"status\")), round(validity_result.get(\"tf1_share\", {}).get(\"point_estimate\", float(\"nan\")), 3)),\n    (\"pipeline_validity.unconditioned_survival_rate\", validity_result.get(\"unconditioned_survival_rate\", {}).get(\"flag\", validity_result.get(\"unconditioned_survival_rate\", {}).get(\"status\")), round(validity_result.get(\"unconditioned_survival_rate\", {}).get(\"point_estimate\", float(\"nan\")), 3)),\n    (\"calibration\", calib_result.get(\"status\"), f\"AUC={calib_result.get('auc', {}).get('point', float('nan')):.3f}\" if calib_result.get(\"status\") == \"COMPUTED\" else \"-\"),\n]\nsummary_df = pd.DataFrame(summary_rows, columns=[\"check\", \"status_or_flag\", \"key_value\"])\nprint(summary_df.to_string(index=False))\n\nfig, axes = plt.subplots(1, 2, figsize=(12, 4.5))\n\n# Left: true vs placebo survival lift by seed\nax = axes[0]\nif placebo_result.get(\"status\") == \"COMPUTED\":\n    seeds = [str(s[\"seed\"]) for s in placebo_result[\"placebo_survival_lift_by_seed\"]]\n    placebo_pts = [s[\"placebo_lift\"] for s in placebo_result[\"placebo_survival_lift_by_seed\"]]\n    placebo_los = [s[\"ci_lo\"] for s in placebo_result[\"placebo_survival_lift_by_seed\"]]\n    placebo_his = [s[\"ci_hi\"] for s in placebo_result[\"placebo_survival_lift_by_seed\"]]\n    x = np.arange(len(seeds))\n    ax.errorbar(x, placebo_pts, yerr=[np.array(placebo_pts) - np.array(placebo_los), np.array(placebo_his) - np.array(placebo_pts)],\n                fmt=\"o\", color=\"tab:gray\", capsize=4, label=\"placebo lift (by seed)\")\n    true_pt = placebo_result[\"true_window_survival_lift\"][\"point\"]\n    true_lo, true_hi = placebo_result[\"true_window_survival_lift\"][\"ci95\"]\n    ax.axhline(true_pt, color=\"tab:red\", linestyle=\"--\", label=\"true-window lift\")\n    ax.axhspan(true_lo, true_hi, color=\"tab:red\", alpha=0.1)\n    ax.axhline(0, color=\"black\", linewidth=0.8)\n    ax.set_xticks(x)\n    ax.set_xticklabels(seeds)\n    ax.set_xlabel(\"seed\")\n    ax.set_ylabel(\"survival-rate lift (high-diffusion minus low-diffusion)\")\n    ax.set_title(f\"Placebo falsification -- verdict: {placebo_result['verdict']}\")\n    ax.legend()\nelse:\n    ax.text(0.5, 0.5, \"placebo_test UNAVAILABLE\", ha=\"center\", va=\"center\")\n    ax.set_axis_off()\n\n# Right: calibration curve\nax = axes[1]\nif calib_result.get(\"status\") == \"COMPUTED\":\n    curve = calib_result[\"calibration_curve_deciles\"]\n    mean_pred = [c[\"mean_predicted\"] for c in curve]\n    obs_rate = [c[\"observed_rate\"] for c in curve]\n    ax.plot([0, 1], [0, 1], linestyle=\"--\", color=\"black\", linewidth=0.8, label=\"perfect calibration\")\n    ax.scatter(mean_pred, obs_rate, color=\"tab:blue\")\n    ax.plot(mean_pred, obs_rate, color=\"tab:blue\", alpha=0.5)\n    ax.set_xlabel(\"mean predicted probability (decile)\")\n    ax.set_ylabel(\"observed survival rate\")\n    ax.set_title(f\"Calibration curve -- Brier={calib_result['brier_score']:.3f}, AUC={calib_result['auc']['point']:.3f}\")\n    ax.legend()\nelse:\n    ax.text(0.5, 0.5, \"calibration UNAVAILABLE\", ha=\"center\", va=\"center\")\n    ax.set_axis_off()\n\nplt.tight_layout()\nplt.show()